In [11]:
import copy
import sys
import os

# Add the project root (i.e., the parent of 'experiments/' and 'src_torch/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [12]:
import copy
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser, L1Regulariser

import torch 
import numpy as np
from src.data_utils import EmbeddingDatasetExtractor
%load_ext autoreload
%autoreload 2
import sys
from abstract_gradient_training.bounded_models import IntervalBoundedModel

import os
import sklearn
from src.interval_utils import get_balanced_min_acc, _get_min_acc
from torch.utils.data import DataLoader

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, FisherTrainer, SimpleTrainer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
DATASET_EXTRACTOR = EmbeddingDatasetExtractor("data/frozen_embeddings")
VALIDATION_RATIO = 0.1
MINIMUM_ITER_FINETUNING = 250000
initial_dataset_name = "jigsaw"

In [14]:
HYPERPARAMETERS = {
    "default" : {
        "min_acc_limit" : 0.83,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.1,
        "checkpoint" : 20, "dual_learning_rate" : 0.1,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.0
    },
    "m2-bert-80M-2k-retrieval" : {
        "min_acc_limit" : 0.76,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.03,
        "checkpoint" : 20, "dual_learning_rate" : 0.001,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.01
    },
    "mxbai-embed-large-v1" : {
        "min_acc_limit" : 0.76,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024, # was 64 
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.03,
        "checkpoint" : 20, "dual_learning_rate" : 0.001,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.0, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.01
    }
}

In [ ]:
device = torch.device("cuda")
def get_model(input_dim: int, seed=0, device="cuda", output_dim=10, head=None):
    """Get a simple CNN model."""
    torch.manual_seed(seed)
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, output_dim),
    ).to(device)
    if head is not None:
        model.append(head)
    return model

def first_head_train_unconstrained(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    


    trainer = SimpleTrainer(
        model, 
    )
    
    result_model = trainer.train(
        dataset_train,
        dataset_val,
        n_epochs = n_epochs,
        batchsize = 64,
        learning_rate = 0.0005,
        weight_decay = 0.0,
    )
    return copy.deepcopy(result_model)


def get_batch(dataset, seed=0, device="cuda", batchsize=100, domain_map_fn=None):
    """Get a batch of data from the dataset."""
    torch.manual_seed(seed)
    dl = torch.utils.data.DataLoader(dataset, batch_size=batchsize)
    batch, labels = next(iter(dl))
    batch, labels = batch.to(device), labels.to(device)
    labels = domain_map_fn(labels) if domain_map_fn else labels
    return batch, labels

def evaluate(model: torch.nn.Sequential, dataset_name: str, llm_name: str):
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X), dim=1)
        accuracy = ((preds == y).sum()/y.shape[0]).item()
        balanced_accuracy = sklearn.metrics.balanced_accuracy_score(
            y.cpu().numpy(), preds.cpu().numpy()
        )
    return accuracy, balanced_accuracy

def certify_balanced_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = get_balanced_min_acc(
            bound_model, X, y
        )
    return certified_balanced_accuracy
def certify_raw_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = _get_min_acc(
            bound_model, X, y, soft = False
        )
    return certified_balanced_accuracy.item()
def finetune_head(trainer: IntervalTrainer, dataset_name: str, llm_name: str, balance: bool = True):
    print("-"*10, f"Finetuning {llm_name} on dataset {dataset_name}", "-"*10)
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    trainer._last_projection = None 
    model_copy = copy.deepcopy(trainer.model)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = n_epochs,
        batch_size = 64,
        lr = 0.0005,
        weight_decay = 0.000,
    )
    result_model = copy.deepcopy(trainer.model)
    trainer.model = model_copy # restore model later 
    return result_model

def first_head_train(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    hyperparameters = None 
    if llm_name in HYPERPARAMETERS: 
        hyperparameters = HYPERPARAMETERS[llm_name]
    else:
        hyperparameters = HYPERPARAMETERS["default"]
    
    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])


    trainer = IntervalTrainer(
        model, 
        min_acc_limit=hyperparameters["min_acc_limit"],
        projection_strategy = hyperparameters["projection_strategy"],
        n_certificate_samples = hyperparameters["n_certificate_samples"],
        batch_size = hyperparameters["rashomon_batchsize"],
        n_iters = hyperparameters["n_iters"],
        obj_alpha=hyperparameters["obj_alpha"], 
        primal_learning_rate=hyperparameters["primal_learning_rate"],
        checkpoint = hyperparameters["checkpoint"], 
        dual_learning_rate=hyperparameters["dual_learning_rate"],
        paradigm="CIL"
    )
    
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = hyperparameters["n_epochs"],
        batch_size = hyperparameters["batchsize"],
        lr = hyperparameters["learning_rate"],
        weight_decay = hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500
    )

    trainer.compute_rashomon_set(dataset_train)
    return trainer, copy.deepcopy(trainer.model)

def print_acc_dict(accuracies_arg):
    for model, accs in accuracies_arg.items():
        print("===> ", model)
        for dataset, acc in accs.items():
            print(f"    |--> Dataset {dataset}")
            print(f"        |--> Accuracies Initial Dataset {initial_dataset_name} (SD/Macro)  : {acc[0]}")
            print(f"        |--> Accuracies Fine-tune dataset (SD/Macro)  : {acc[1]}")

### Reproducibility
To get the same results as in the paper all cells should be run one after the other in the order of this notebook. 

In [16]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [17]:
llm_names = ["text-embedding-3-large", "mxbai-embed-large-v1", "m2-bert-80M-2k-retrieval", "gte-large", "voyage-3"]
print("*"*10, f"Initial training of models on {initial_dataset_name}", "*"*10)
trainers = {}
original_models = {}
for llm_name in llm_names:
    print("-"*10, f"Initial training of {llm_name} on {initial_dataset_name}", "-"*10)
    trainers[llm_name], original_models[llm_name] = first_head_train(
        initial_dataset_name, llm_name, balance=True # Jigsaw is heavily unbalanced. Duplicating ones for the initial training reduces variance inside the same batch. 
    )

********** Initial training of models on jigsaw **********
---------- Initial training of text-embedding-3-large on jigsaw ----------
Dataset not found or cache not used, extracting it now.


Training Epochs: 100%|█████████████████████████████████████████████| 1/1 [00:32<00:00, 32.26s/it, val_loss=0.1653, val_acc=0.9346, proj=None, progress=1.00]


Initial acc constraint violation: -0.1165 (Positive = violated)
Number of model parameters: 6146
Computing Rashomon set with min acc limit: 0.83
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.95,  Min acc soft=0.95


100%|█████████████████████████████████████████████████████████████████████████| 700/700 [00:35<00:00, 19.73it/s, size=261.29, obj=0.043, min_soft_acc=0.773]


Final bbox:  Obj=0.04,  Size=261.29,  Min acc hard=0.80,  Min acc soft=0.80
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['11.45', '72.22', '152.22', '229.56', '267.54', '273.82', '272.90', '271.91', '271.36', '270.84', '270.34', '269.87', '269.42', '268.98', '268.56', '268.16', '267.75', '267.34', '266.93', '266.54', '266.16', '265.81', '265.44', '265.08', '264.72', '264.36', '264.00', '263.65', '263.30', '262.96', '262.62', '262.28', '261.95', '261.62', '261.29']
Checkpoint certificates: ['0.94', '0.92', '0.87', '0.81', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.79', '0.80', '0.80', '0.80', '0.80', '0.80', '0.80', '0.80', '0.80']
----------------------- Finished Computing Rashomon set ------------------------
---------- Initial training of mxbai-embed-large-v1 on jigs

...

Final bbox:  Obj=0.04,  Size=82.29,  Min acc hard=0.77,  Min acc soft=0.76
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['11.45', '71.06', '94.94', '94.24', '93.20', '92.74', '92.28', '91.83', '91.40', '90.98', '90.58', '90.19', '89.80', '89.42', '89.05', '88.69', '88.33', '87.98', '87.63', '87.27', '86.92', '86.58', '86.25', '85.91', '85.58', '85.25', '84.92', '84.59', '84.26', '83.92', '83.59', '83.26', '82.93', '82.61', '82.29']
Checkpoint certificates: ['0.88', '0.77', '0.73', '0.74', '0.74', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.75', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.76', '0.77', '0.77', '0.77', '0.77']
----------------------- Finished Computing Rashomon set ------------------------
---------- Initial training of voyage-3 on jigsaw ----------
Dataset not found or cache not u

Training Epochs: 100%|█████████████████████████████████████████████| 1/1 [00:32<00:00, 32.59s/it, val_loss=0.2854, val_acc=0.8808, proj=None, progress=1.00]


Initial acc constraint violation: -0.0450 (Positive = violated)
Number of model parameters: 2050
Computing Rashomon set with min acc limit: 0.83
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.88,  Min acc soft=0.88


100%|██████████████████████████████████████████████████████████████████████████| 700/700 [00:40<00:00, 17.34it/s, size=65.60, obj=0.032, min_soft_acc=0.799]


Final bbox:  Obj=0.03,  Size=65.60,  Min acc hard=0.75,  Min acc soft=0.75
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['11.45', '66.48', '79.36', '77.55', '76.54', '76.09', '75.63', '75.19', '74.77', '74.37', '73.97', '73.58', '73.20', '72.83', '72.46', '72.10', '71.73', '71.37', '71.03', '70.68', '70.33', '69.98', '69.64', '69.30', '68.96', '68.62', '68.27', '67.94', '67.60', '67.27', '66.94', '66.61', '66.27', '65.94', '65.60']
Checkpoint certificates: ['0.86', '0.74', '0.71', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.74', '0.74', '0.74', '0.74', '0.74', '0.74', '0.74', '0.74', '0.74', '0.75', '0.75', '0.75', '0.75', '0.75']
----------------------- Finished Computing Rashomon set ------------------------


In [18]:
no_finetune_accuracies = {}
finetune_dataset_names = ["tweets_hate_speech_detection", "civil_comments", "hatemoji", "sbic", "hatecheck"]
for llm_name_to_finetune in llm_names: 
    no_finetune_accuracies[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        no_finetune_model = original_models[llm_name_to_finetune]
        accs_new_dataset = evaluate(
            no_finetune_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            no_finetune_model, initial_dataset_name, llm_name_to_finetune
        )
        no_finetune_accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
print("*"*20, "Initial results with no fine-tuning", "*"*20)
print_acc_dict(no_finetune_accuracies)
#9244218468666077

Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting it now.
Dataset not found or cache not used, extracting 

In [ ]:
accuracies = {}
certificates = {}
for llm_name_to_finetune in llm_names: 
    accuracies[llm_name_to_finetune] = {}
    certificates[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        current_trainer: IntervalTrainer = trainers[llm_name_to_finetune]
        finetuned_model = finetune_head(
            current_trainer, finetune_dataset_name, llm_name_to_finetune, balance=True
        )
        
        accs_new_dataset = evaluate(
            finetuned_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            finetuned_model, initial_dataset_name, llm_name_to_finetune
        )
        accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
        
        current_outer_bbox = current_trainer.get_current_bbox()
        certified_balanced_accuracy = certify_balanced_accuracy(
            current_outer_bbox, initial_dataset_name, llm_name_to_finetune
        )
        certificates[llm_name_to_finetune][finetune_dataset_name] = certified_balanced_accuracy
        print(f"=> Done ! Certified balanced accuracy {certified_balanced_accuracy}")
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
print("-"*20, "Final Results with Certified Fine-Tuning","-"*20)
print_acc_dict(accuracies)

---------- Finetuning text-embedding-3-large on dataset tweets_hate_speech_detection ----------


Training Epochs: 100%|████████████████████████████████████████████████| 4/4 [03:02<00:00, 45.66s/it, val_loss=0.5749, val_acc=0.7289, proj=3, progress=1.00]


=> Done ! Certified balanced accuracy 0.8069323587035833
    |--> Accuracies on new dataset (SD/Macro) (0.8316646218299866, 0.7229399701715138)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.9157109260559082, 0.9354647865069583)
---------- Finetuning text-embedding-3-large on dataset civil_comments ----------


Training Epochs: 100%|███████████████████████████████████████████████| 1/1 [03:04<00:00, 184.29s/it, val_loss=0.3992, val_acc=0.8236, proj=4, progress=1.00]


=> Done ! Certified balanced accuracy 0.7762395609659616
    |--> Accuracies on new dataset (SD/Macro) (0.8272384405136108, 0.8253605204417294)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.9299993515014648, 0.9364273571749868)
---------- Finetuning text-embedding-3-large on dataset hatemoji ----------


...

Training Epochs: 100%|██████████████████████████████████████████████| 26/26 [01:13<00:00,  2.84s/it, val_loss=0.8136, val_acc=0.5630, proj=6, progress=0.99]


=> Done ! Certified balanced accuracy 0.7313255309542865
    |--> Accuracies on new dataset (SD/Macro) (0.614503800868988, 0.5670993428453022)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.8517264723777771, 0.8379133504191192)
---------- Finetuning m2-bert-80M-2k-retrieval on dataset sbic ----------


Training Epochs: 100%|████████████████████████████████████████████████| 8/8 [01:20<00:00, 10.07s/it, val_loss=0.7069, val_acc=0.6449, proj=1, progress=1.00]


=> Done ! Certified balanced accuracy 0.7232368356412497
    |--> Accuracies on new dataset (SD/Macro) (0.6474470496177673, 0.6556578951258378)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.861064076423645, 0.8337391277013665)
---------- Finetuning m2-bert-80M-2k-retrieval on dataset hatecheck ----------


Training Epochs:   4%|█▋                                             | 1/27 [00:03<01:36,  3.71s/it, val_loss=0.8624, val_acc=0.5651, proj=1, progress=0.02]

In [ ]:
with open('llm_final_results.txt', 'w') as f:
    print("*"*10, "Final Results", "*"*10, file=f)
    print("Model ", " & ".join([finetune_dataset_name for finetune_dataset_name in finetune_dataset_names]), file=f)
    for llm_name in llm_names:
        line = llm_name + " & "
        accuracy = accuracies[llm_name]
        for finetune_dataset_name in finetune_dataset_names: 
            certificate = certificates[llm_name][finetune_dataset_name]
            acc = accuracy[finetune_dataset_name]
            line += f"{round(acc[0][1], 2)}({round(acc[0][0], 2)})"
            line += f"/{round(acc[1][1], 2)}({round(acc[1][0], 2)})"
            line += f"[{round(certificate, 2)}]  & "
        print(line, file=f)

### FisherTrainer

In [6]:
def fisher_first_head_train(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    hyperparameters = None 
    if llm_name in HYPERPARAMETERS: 
        hyperparameters = HYPERPARAMETERS[llm_name]
    else:
        hyperparameters = HYPERPARAMETERS["default"]
    
    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])


    trainer = FisherTrainer(
        model, 
        min_acc_limit=hyperparameters["min_acc_limit"],
        projection_strategy = hyperparameters["projection_strategy"],
        n_certificate_samples = hyperparameters["n_certificate_samples"],
        batch_size = hyperparameters["rashomon_batchsize"],
        n_iters = hyperparameters["n_iters"],
        obj_alpha=hyperparameters["obj_alpha"], 
        primal_learning_rate=hyperparameters["primal_learning_rate"],
        checkpoint = hyperparameters["checkpoint"], 
        dual_learning_rate=hyperparameters["dual_learning_rate"],
        paradigm="CIL"
    )
    
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = hyperparameters["n_epochs"],
        batch_size = hyperparameters["batchsize"],
        lr = hyperparameters["learning_rate"],
        weight_decay = hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500
    )

    return trainer, copy.deepcopy(trainer.model)

def fisher_finetune_head(trainer: FisherTrainer, dataset_name: str, original_dataset_name: str, llm_name: str, balance: bool = True):
    print("-"*10, f"Finetuning {llm_name} on dataset {dataset_name}", "-"*10)
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )

    og_dataset_train, og_dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, original_dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )

    trainer.compute_rashomon_set(
        dataset_val,
        og_dataset_val,
        prune_prop=0.8,
        fisher_batch_size=500,
        fisher_epochs=200,
        use_outer_bbox=False
    )

    trainer._last_projection = None 
    model_copy = copy.deepcopy(trainer.model)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = n_epochs,
        batch_size = 64,
        lr = 0.0005,
        weight_decay = 0.000,
    )
    result_model = copy.deepcopy(trainer.model)
    trainer._last_projection = None
    trainer.bounds = []
    trainer.certificates = []
    trainer.final_certificates = []
    trainer.model = model_copy # restore model later
    return result_model

### Fisher Experiments

In [7]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [8]:
# llm_names = ["text-embedding-3-large", "mxbai-embed-large-v1", "m2-bert-80M-2k-retrieval", "gte-large", "voyage-3"]
llm_names = ["text-embedding-3-large"]
print("*"*10, f"Initial training of models on {initial_dataset_name}", "*"*10)
trainers = {}
original_models = {}
for llm_name in llm_names:
    print("-"*10, f"Initial training of {llm_name} on {initial_dataset_name}", "-"*10)
    trainers[llm_name], original_models[llm_name] = fisher_first_head_train(
        initial_dataset_name, llm_name, balance=True # Jigsaw is heavily unbalanced. Duplicating ones for the initial training reduces variance inside the same batch. 
    )

********** Initial training of models on jigsaw **********
---------- Initial training of text-embedding-3-large on jigsaw ----------
Dataset not found or cache not used, extracting it now.


/data/le24/CertifiedContinualLearning/src/data_utils.py:356: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  return torch.from_numpy(self._features[idx]), torch.tensor(self._labels[idx])
Training Epochs: 100%|█████████████████████████████████████████████| 1/1 [00:32<00:00, 32.73s/it, val_loss=0.1653, val_acc=0.9346, proj=None, progress=1.00]


In [10]:
finetune_dataset_names = ["tweets_hate_speech_detection", "civil_comments", "hatemoji", "sbic", "hatecheck"]
accuracies = {}
certificates = {}
for llm_name_to_finetune in llm_names: 
    accuracies[llm_name_to_finetune] = {}
    certificates[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names[0:1]:
        current_trainer: FisherTrainer = trainers[llm_name_to_finetune]
        finetuned_model = fisher_finetune_head(
            current_trainer, finetune_dataset_name, initial_dataset_name, llm_name_to_finetune, balance=True
        )
        
        accs_new_dataset = evaluate(
            finetuned_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            finetuned_model, initial_dataset_name, llm_name_to_finetune
        )
        accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
        
        current_outer_bbox = current_trainer.get_current_bbox()
        certified_balanced_accuracy = certify_balanced_accuracy(
            current_outer_bbox, initial_dataset_name, llm_name_to_finetune
        )
        certificates[llm_name_to_finetune][finetune_dataset_name] = certified_balanced_accuracy
        print(f"=> Done ! Certified balanced accuracy {certified_balanced_accuracy}")
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
print("-"*20, "Final Results with Certified Fine-Tuning","-"*20)
print_acc_dict(accuracies)

---------- Finetuning text-embedding-3-large on dataset tweets_hate_speech_detection ----------
Dataset not found or cache not used, extracting it now.
Parameter importance sum: 0.20503824949264526
Non zero params: 6146
To keep top 20%, found global Fisher threshold: 0.00002251
Remaining number of parameters: tensor(4916, device='cuda:0')
Initial acc constraint violation: -0.0437 (Positive = violated)
Number of model parameters: 6146
Computing Rashomon set with min acc limit: 0.64
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.68,  Min acc soft=0.68


100%|███████████████████████████████████████████████████████████████████████████| 700/700 [00:37<00:00, 18.67it/s, size=0.01, obj=0.000, min_soft_acc=0.467]


Final bbox:  Obj=0.00,  Size=0.01,  Min acc hard=0.68,  Min acc soft=0.68
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['0.06', '0.05', '0.04', '0.03', '0.02', '0.02', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01']
Checkpoint certificates: ['0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68', '0.68']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:  75%|███████████████████████████████████▎           | 3/4 [02:03<00:41, 41.18s/it, val_loss=0.6975, val_acc=0.6819, proj=34, progress=0.37]


KeyboardInterrupt: 

### Baseline